In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS catalog.silver;

In [0]:
%sql
SELECT * FROM catalog.bronze.physicians_raw;

In [0]:
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

In [0]:
bronze_table = 'catalog.bronze.physicians_raw'
silver_table = 'catalog.silver.dim_physicians'
checkpoint_path = "abfss://data@datastoragea.dfs.core.windows.net/silver/dim_physicians/checkpoint/"

df_physicians_bronze = spark.readStream.table(bronze_table)

df_physicians_clean = (
    df_physicians_bronze
        .drop(
            "_rescued_data",
            "_source_file",
            "_ingest_timestamp",
            "ingestion_date"
        )
        .withColumnRenamed("full_name", "physician_name")
        .withColumn("load_timestamp", current_timestamp())
)

def merge_dim_physicians(batch_df, batch_id):
    batch_df = batch_df.dropDuplicates(["physician_id"])
    
    if not spark.catalog.tableExists(silver_table):
        # First run — create the table
        (batch_df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(silver_table))  
        return
    
    # Load Delta table by name and upsert into Silver
    dim_physicians = DeltaTable.forName(spark, silver_table)

    (dim_physicians.alias("t")
        .merge(
            batch_df.alias("s"),
            "t.physician_id = s.physician_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute())
    

In [0]:
(   
    df_physicians_clean.writeStream
        .foreachBatch(merge_dim_physicians)  # for every batch, merge_dim_physicians is run
        .outputMode("update")
        .trigger(availableNow=True)
        .option("checkpointLocation", checkpoint_path)
        .start()
)

In [0]:
%sql
SELECT * FROM catalog.silver.dim_physicians;

In [0]:
# Data quality check
from pyspark.sql.functions import col, count

print("DIM PHYSICIANS")
df = spark.read.table("catalog.silver.dim_physicians")
total = df.count()
print(f"Total rows: {total} (expected: 50)")

dup_id = total - df.select("physician_id").distinct().count()
print(f"Duplicate physician_ids: {'OK' if dup_id == 0 else dup_id}")

for c in ["physician_id", "physician_name", "specialization",
          "assigned_branch", "physician_type"]:
    n = df.filter(col(c).isNull()).count()
    print(f"  {'OK' if n == 0 else 'CHECK'} {c}: {n} nulls")

print("Specialization distribution:")
df.groupBy("specialization").count().orderBy("count", ascending=False).show()

print("Physician type distribution:")
df.groupBy("physician_type").count().show()

# Assigned branch should all be valid: H001-H010
invalid_branch = df.filter(
    ~col("assigned_branch").isin([f"H{i:03d}" for i in range(1,11)])
).count()
print(f"Invalid assigned_branch: {'OK' if invalid_branch == 0 else f'CHECK — {invalid_branch} rows'}")